# 보이스피싱 독립 모델 교차검증

기존 Whisper + pyannote + 키워드 규칙 결과를 서로 다른 계열의 모델로 재평가합니다.

- 2차 전사: `facebook/mms-1b-all` 한국어 어댑터 (Wav2Vec2 CTC)
- 2차 화자: `speechbrain/spkrec-ecapa-voxceleb` 임베딩 + 계층 군집화
- 2차 역할: `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` 다국어 NLI
- 검수 선정: 불일치·빈 전사·다중 사건 우선 + 일치 발화 고정 난수 10%, 총 300~500개

이 결과는 정답 정확도가 아니라 **모델 간 일치도**입니다. 최종 정확도는 자동 추출된 300~500개를 사람이 확인한 뒤 계산합니다. MMS 모델 라이선스는 CC BY-NC 4.0이므로 비상업 연구 검증에 사용하며 상업적 이용은 별도 검토해야 합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install 'transformers>=4.45' accelerate sentencepiece speechbrain librosa soundfile jiwer scikit-learn openpyxl pandas scipy
from pathlib import Path
import gc, json, math, os, random, re, warnings
import numpy as np, pandas as pd, torch, librosa
warnings.filterwarnings('ignore')
assert torch.cuda.is_available(), '런타임 유형을 GPU로 변경하세요.'
print('GPU:',torch.cuda.get_device_name(0))

## 1. 경로와 실행 범위

기존 `검수_채점표.xlsx`의 표본 50개를 사용합니다. 원본 및 결과 경로가 앞선 노트북과 같으면 수정할 필요가 없습니다.

In [ ]:
DRIVE=Path('/content/drive/MyDrive')
BASE=DRIVE/'보이스피싱_분석'
RESULT_ROOT=BASE/'분석 결과'
AUDIO_ROOT=BASE/'원본 영상 및 음원'
REVIEW_ROOT=BASE/'검수 및 보고서'
SCORECARD=REVIEW_ROOT/'검수_채점표.xlsx'
OUT=BASE/'독립 모델 교차검증'
CACHE=OUT/'cache'; OUT.mkdir(parents=True,exist_ok=True); CACHE.mkdir(parents=True,exist_ok=True)
MIN_REVIEW=300; MAX_REVIEW=500; AGREEMENT_SAMPLE_RATE=.10; SEED=20260812
assert SCORECARD.exists(), f'채점표가 없습니다: {SCORECARD}'
sample_files=pd.read_excel(SCORECARD,sheet_name='표본파일')
first_turns=pd.read_excel(SCORECARD,sheet_name='발화검수')
first_cases=pd.read_excel(SCORECARD,sheet_name='사건검수')
print('표본 파일:',sample_files.file_id.nunique(),'1차 발화:',len(first_turns),'1차 사건:',len(first_cases))

## 2. 원본 경로 연결 및 오디오 캐시

MMS 전사는 1차 발화 시간 구간을 잘라 독립적으로 다시 인식합니다. 따라서 전사 모델은 독립적이지만 발화 구간 자체는 1차 결과를 사용한다는 제한이 있습니다.

In [ ]:
media={p.name:p for p in AUDIO_ROOT.rglob('*') if p.suffix.lower() in {'.mp3','.mp4'}}
def audio_path(source):
    p=AUDIO_ROOT/str(source)
    return p if p.exists() else media.get(Path(str(source)).name)
missing=[s for s in sample_files.source_file if audio_path(s) is None]
assert not missing, f'표본 원본 {len(missing)}개를 찾지 못했습니다. AUDIO_ROOT를 확인하세요.'
def load_audio(source):
    key=re.sub(r'[^0-9A-Za-z가-힣_-]+','_',Path(str(source)).stem)[:90]
    cp=CACHE/f'{key}.npy'
    if cp.exists(): return np.load(cp,mmap_mode='r')
    y,_=librosa.load(str(audio_path(source)),sr=16000,mono=True)
    np.save(cp,y.astype('float32')); return np.load(cp,mmap_mode='r')
print('표본 원본 연결 정상')

## 3. 독립 ASR — Meta MMS

완료 결과를 `mms_asr.jsonl`에 즉시 저장하므로 세션이 끊기면 같은 셀을 다시 실행하면 됩니다. 지나치게 짧은 구간은 앞뒤 0.15초를 포함합니다.

In [ ]:
from transformers import AutoProcessor, Wav2Vec2ForCTC
MMS_ID='facebook/mms-1b-all'
processor=AutoProcessor.from_pretrained(MMS_ID)
mms=Wav2Vec2ForCTC.from_pretrained(MMS_ID,torch_dtype=torch.float16,low_cpu_mem_usage=True).to('cuda').eval()
processor.tokenizer.set_target_lang('kor'); mms.load_adapter('kor'); mms.to('cuda').eval()
mms_path=OUT/'mms_asr.jsonl'
done={}
if mms_path.exists():
    for line in mms_path.read_text(encoding='utf-8').splitlines():
        x=json.loads(line); done[x['key']]=x
def norm_text(x): return re.sub(r'[^0-9A-Za-z가-힣]+','',str(x or '')).lower()
for n,r in enumerate(first_turns.itertuples(index=False),1):
    key=f'{r.file_id}|{r.case_id}|{r.turn_id}'
    if key in done: continue
    y=load_audio(r.source_file); a=max(0,int((float(r.start)-.15)*16000)); b=min(len(y),int((float(r.end)+.15)*16000)); clip=np.asarray(y[a:b],dtype='float32')
    if len(clip)<800: text2=''
    else:
        inp=processor(clip,sampling_rate=16000,return_tensors='pt').input_values.to('cuda',dtype=torch.float16)
        with torch.inference_mode(): ids=torch.argmax(mms(inp).logits,dim=-1)[0].cpu()
        text2=processor.decode(ids).strip()
    item={'key':key,'second_text':text2}; done[key]=item
    with mms_path.open('a',encoding='utf-8') as f: f.write(json.dumps(item,ensure_ascii=False)+'\n')
    if n%100==0: print(n,'/',len(first_turns))
print('MMS 완료:',len(done))

## 4. 독립 화자 — SpeechBrain ECAPA + 군집화

각 발화 임베딩을 만들고 사건별로 1~4명 화자를 추정합니다. 화자 ID는 사건별 최적 매핑 후 1차 결과와 비교합니다.

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
ecapa=EncoderClassifier.from_hparams(source='speechbrain/spkrec-ecapa-voxceleb',savedir=str(CACHE/'ecapa'),run_opts={'device':'cuda'})
spk_path=OUT/'ecapa_speakers.jsonl'; spk_done={}
if spk_path.exists():
    for line in spk_path.read_text(encoding='utf-8').splitlines(): x=json.loads(line); spk_done[x['key']]=x
for (fid,cid),g in first_turns.groupby(['file_id','case_id'],sort=False):
    keys=[f'{r.file_id}|{r.case_id}|{r.turn_id}' for r in g.itertuples(index=False)]
    if all(k in spk_done for k in keys): continue
    embs=[]
    for r in g.itertuples(index=False):
        y=load_audio(r.source_file); a=max(0,int(float(r.start)*16000)); b=min(len(y),int(float(r.end)*16000)); clip=np.asarray(y[a:b],dtype='float32')
        if len(clip)<16000: clip=np.pad(clip,(0,16000-len(clip)))
        wav=torch.from_numpy(clip).unsqueeze(0).to('cuda')
        with torch.inference_mode(): e=ecapa.encode_batch(wav).squeeze().float().cpu().numpy()
        embs.append(e/(np.linalg.norm(e)+1e-9))
    X=np.stack(embs); best=np.zeros(len(X),dtype=int); best_score=-2
    for k in range(2,min(4,len(X)-1)+1):
        lab=AgglomerativeClustering(n_clusters=k,metric='cosine',linkage='average').fit_predict(X)
        if len(set(lab))>1:
            score=silhouette_score(X,lab,metric='cosine')
            if score>best_score: best,best_score=lab,score
    for key,label in zip(keys,best):
        item={'key':key,'second_speaker_id':f'ECAPA_{int(label):02d}','cluster_score':float(best_score)}; spk_done[key]=item
        with spk_path.open('a',encoding='utf-8') as f: f.write(json.dumps(item,ensure_ascii=False)+'\n')
print('ECAPA 완료:',len(spk_done)); del ecapa; torch.cuda.empty_cache(); gc.collect()

## 5. 독립 역할 — 다국어 NLI

화자별 2차 전사 문맥을 모아 범인/피해자/제3자 가설의 함의 점수를 비교합니다. 기존 키워드 목록은 사용하지 않습니다.

In [ ]:
if 'mms' in globals():
    del mms
torch.cuda.empty_cache(); gc.collect()
from transformers import pipeline
NLI_ID='MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
# Colab의 일부 PyTorch/CUDA 조합에서 DeBERTa 상대위치 연산 오류가 발생하므로 NLI는 CPU float32로 실행한다.
nli=pipeline('zero-shot-classification',model=NLI_ID,device=-1,torch_dtype=torch.float32)
mms_df=pd.DataFrame(done.values()); spk_df=pd.DataFrame(spk_done.values())
base=first_turns.copy(); base['key']=base.apply(lambda r:f"{r.file_id}|{r.case_id}|{r.turn_id}",axis=1)
base=base.merge(mms_df,on='key',how='left').merge(spk_df,on='key',how='left')
role_map={}; role_path=OUT/'nli_roles.jsonl'
if role_path.exists():
    for line in role_path.read_text(encoding='utf-8').splitlines(): x=json.loads(line); role_map[x['key']]=x
labels_ko=['보이스피싱을 시도하며 돈이나 개인정보를 요구하는 범인','전화를 받고 질문하거나 거절하는 피해자','안내방송 또는 사건과 무관한 제3자']
role_names={'보이스피싱을 시도하며 돈이나 개인정보를 요구하는 범인':'OFFENDER','전화를 받고 질문하거나 거절하는 피해자':'VICTIM','안내방송 또는 사건과 무관한 제3자':'THIRD_PARTY'}
for (fid,cid,sid),g in base.groupby(['file_id','case_id','second_speaker_id'],dropna=False):
    rk=f'{fid}|{cid}|{sid}'
    if rk in role_map: continue
    context=' '.join(g.second_text.fillna('').astype(str).tolist())[:3500]
    if not context.strip(): item={'key':rk,'second_role':'REVIEW','role_nli_score':0.0}
    else:
        z=nli(context,labels_ko,multi_label=False,hypothesis_template='이 화자는 {}이다.')
        item={'key':rk,'second_role':role_names[z['labels'][0]],'role_nli_score':float(z['scores'][0])}
    role_map[rk]=item
    with role_path.open('a',encoding='utf-8') as f: f.write(json.dumps(item,ensure_ascii=False)+'\n')
base['role_key']=base.apply(lambda r:f"{r.file_id}|{r.case_id}|{r.second_speaker_id}",axis=1)
roles_df=pd.DataFrame(role_map.values()).rename(columns={'key':'role_key'})
base=base.merge(roles_df,on='role_key',how='left')
print('NLI 역할 완료:',len(role_map))

## 6. 모델 간 비교

화자 ID 이름은 임의이므로 사건별 헝가리안 최적 매핑 후 일치 여부를 계산합니다. 전사는 문자 정규화 후 발화별 CER을 계산합니다.

In [ ]:
from jiwer import cer
from scipy.optimize import linear_sum_assignment
base['intermodel_cer']=base.apply(lambda r:cer(str(r.auto_text or ''),str(r.second_text or '')) if str(r.auto_text or '').strip() else (0.0 if not str(r.second_text or '').strip() else 1.0),axis=1)
base['role_agree']=base.auto_role.fillna('REVIEW').eq(base.second_role.fillna('REVIEW'))
base['mapped_second_speaker']=''
for (fid,cid),g in base.groupby(['file_id','case_id']):
    a=sorted(g.auto_speaker_id.fillna('').unique()); b=sorted(g.second_speaker_id.fillna('').unique()); mat=np.zeros((len(a),len(b)))
    for _,r in g.iterrows(): mat[a.index(r.auto_speaker_id if pd.notna(r.auto_speaker_id) else ''),b.index(r.second_speaker_id if pd.notna(r.second_speaker_id) else '')]+=max(.001,float(r.duration_sec))
    rr,cc=linear_sum_assignment(-mat); mapping={b[j]:a[i] for i,j in zip(rr,cc)}
    base.loc[g.index,'mapped_second_speaker']=g.second_speaker_id.map(mapping).fillna('UNMAPPED')
base['speaker_agree']=base.auto_speaker_id.eq(base.mapped_second_speaker)
base['asr_agree']=base.intermodel_cer.le(.10)
base['any_disagree']=~(base.asr_agree & base.role_agree & base.speaker_agree)
base['multi_case']=base.file_id.isin(set(sample_files.loc[sample_files.case_count.gt(1),'file_id']))
comparison_path=OUT/'50개_독립모델_교차검증.xlsx'
with pd.ExcelWriter(comparison_path,engine='openpyxl') as w:
    base.to_excel(w,sheet_name='발화비교',index=False)
    base.groupby('file_id').agg(발화수=('key','size'),전사일치율=('asr_agree','mean'),역할일치율=('role_agree','mean'),화자일치율=('speaker_agree','mean'),평균모델간CER=('intermodel_cer','mean'),다중사건=('multi_case','max')).reset_index().to_excel(w,sheet_name='파일요약',index=False)
print('교차검증 저장:',comparison_path)
print({'전사일치율':base.asr_agree.mean(),'역할일치율':base.role_agree.mean(),'화자일치율':base.speaker_agree.mean(),'불일치발화':int(base.any_disagree.sum())})

## 7. 사람 검수 300~500개 자동 추출

우선순위는 역할 불일치 → 높은 모델 간 CER → 화자 불일치 → 다중사건입니다. 일치 발화도 10% 무작위로 포함해 두 모델이 함께 틀리는 경우를 추정합니다.

In [ ]:
rng=np.random.default_rng(SEED)
x=base.copy(); x['priority_score']=(~x.role_agree)*4 + x.intermodel_cer.clip(0,2)*2 + (~x.speaker_agree)*2 + x.multi_case*1
dis=x[x.any_disagree].sort_values(['priority_score','intermodel_cer'],ascending=False)
agree=x[~x.any_disagree]
agree_n=min(len(agree),max(int(len(agree)*AGREEMENT_SAMPLE_RATE),50))
agree_sample=agree.sample(n=agree_n,random_state=SEED) if agree_n else agree
selected=pd.concat([dis.head(max(0,MAX_REVIEW-len(agree_sample))),agree_sample]).drop_duplicates('key')
if len(selected)<MIN_REVIEW:
    extra=x[~x.key.isin(selected.key)].sort_values('priority_score',ascending=False).head(MIN_REVIEW-len(selected)); selected=pd.concat([selected,extra])
selected=selected.head(MAX_REVIEW).sort_values(['file_id','case_id','start']).reset_index(drop=True)
selected.insert(0,'review_no',range(1,len(selected)+1))
for col,val in [('검수자',''),('검수상태','미검수'),('정답_텍스트',''),('정답_화자ID',''),('정답_역할',''),('음성변조',''),('검수메모','')]: selected[col]=val
review_path=OUT/'사람검수_300_500개.xlsx'
guide=pd.DataFrame([['목적','독립 모델 불일치와 일치 10% 표본의 사람 정답 생성'],['검수상태','완료된 행만 정량 정확도 계산에 사용'],['정답_역할','OFFENDER / VICTIM / THIRD_PARTY / UNSURE / EXCLUDE'],['주의','두 자동 결과 중 하나를 고르는 것이 아니라 원본 음성을 듣고 독립적으로 정답 입력']],columns=['항목','설명'])
with pd.ExcelWriter(review_path,engine='openpyxl') as w: guide.to_excel(w,sheet_name='안내',index=False); selected.to_excel(w,sheet_name='검수대상',index=False)
from openpyxl import load_workbook
from openpyxl.styles import Font,PatternFill,Alignment
from openpyxl.worksheet.datavalidation import DataValidation
wb=load_workbook(review_path); ws=wb['검수대상']; ws.freeze_panes='A2'; ws.auto_filter.ref=ws.dimensions
for s in wb.worksheets:
    s.sheet_view.showGridLines=False
    for c in s[1]: c.font=Font(bold=True,color='FFFFFF'); c.fill=PatternFill('solid',fgColor='1F4E78')
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=min(45,max(10,max(len(str(c.value or '')) for c in col[:200])+2))
headers={c.value:c.column_letter for c in ws[1]}
for h,vals in [('검수상태',['미검수','검수중','완료','제외']),('정답_역할',['OFFENDER','VICTIM','THIRD_PARTY','UNSURE','EXCLUDE']),('음성변조',['TRUE','FALSE','UNSURE'])]:
    dv=DataValidation(type='list',formula1='"'+','.join(vals)+'"'); ws.add_data_validation(dv); dv.add(f'{headers[h]}2:{headers[h]}1048576')
wb.save(review_path)
print('사람 검수 대상:',len(selected),'개 / 일치 무작위:',len(agree_sample),'개')
print('저장:',review_path)

## 8. 교차검증 보고서 자동 생성

사람 검수 전에는 모델 간 일치도 보고서만 생성합니다. 사람이 `사람검수_300_500개.xlsx`를 완료한 뒤 마지막 셀을 다시 실행하면 실제 정확도 항목도 추가됩니다.

In [ ]:
summary={'표본파일':int(base.file_id.nunique()),'발화':len(base),'전사일치율_CER10이하':float(base.asr_agree.mean()),'역할일치율':float(base.role_agree.mean()),'화자일치율':float(base.speaker_agree.mean()),'평균_모델간_CER':float(base.intermodel_cer.mean()),'사람검수선정':len(selected)}
human_section='사람 검수 미완료'
try:
    h=pd.read_excel(review_path,sheet_name='검수대상'); h=h[h['검수상태'].eq('완료')]
    if len(h):
        from sklearn.metrics import classification_report,accuracy_score
        ht=h[h['정답_텍스트'].fillna('').str.strip().ne('')]; hr=h[h['정답_역할'].isin(['OFFENDER','VICTIM'])]
        cer1=cer(ht.정답_텍스트.astype(str).tolist(),ht.auto_text.astype(str).tolist()) if len(ht) else np.nan
        cer2=cer(ht.정답_텍스트.astype(str).tolist(),ht.second_text.astype(str).tolist()) if len(ht) else np.nan
        acc1=accuracy_score(hr.정답_역할,hr.auto_role.fillna('REVIEW')) if len(hr) else np.nan
        acc2=accuracy_score(hr.정답_역할,hr.second_role.fillna('REVIEW')) if len(hr) else np.nan
        human_section=f'사람 검수 {len(h)}개 기준: 1차 CER {cer1:.4f}, 2차 CER {cer2:.4f}, 1차 역할 정확도 {acc1:.4f}, 2차 역할 정확도 {acc2:.4f}'
except Exception as e: human_section=f'사람 검수 결과 읽기 전: {e}'
md=f'''# 독립 모델 교차검증 보고서

## 모델 구성

- 1차: faster-whisper large-v3-turbo + pyannote community-1 + 키워드 역할 규칙
- 2차: Meta MMS Wav2Vec2 CTC + SpeechBrain ECAPA 군집화 + 다국어 NLI 역할 분류

## 모델 간 일치도

- 표본 파일: {summary['표본파일']}개
- 비교 발화: {summary['발화']:,}개
- 전사 일치율(모델 간 CER ≤ 0.10): {summary['전사일치율_CER10이하']:.2%}
- 역할 일치율: {summary['역할일치율']:.2%}
- 화자 일치율: {summary['화자일치율']:.2%}
- 평균 모델 간 CER: {summary['평균_모델간_CER']:.4f}
- 사람 검수 선정: {summary['사람검수선정']}개

## 사람 정답 기준 결과

{human_section}

## 한계

모델 간 일치는 실제 정답을 보장하지 않는다. 2차 MMS는 CC BY-NC 4.0이며 비상업 연구 검증에 사용한다. 2차 전사는 1차 발화 시간 구간을 사용하므로 발화 세그멘테이션이 완전히 독립적이지 않다. ECAPA 군집 화자 수는 발화 임베딩으로 추정하며 정식 DER이 아니다. 최종 신뢰성 결론은 사람이 확인한 불일치 발화와 일치 무작위 표본에서 계산해야 한다.
'''
rp=OUT/'독립모델_교차검증_보고서.md'; rp.write_text(md,encoding='utf-8')
(OUT/'교차검증_요약.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(md); print('보고서:',rp)